<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/healthstack-logo-dark.png" alt="HealthStack logo" width="320" />
</div>

<p align="center"><strong>AFROTECH 2026</strong></p>
<p align="center"><font size="6"><strong>Building Fairness Into the Data Supply Chain</strong></font></p>
<p align="center"><strong>Instructor: Waverly Rose Brim</strong></p>

---

<p align="center"><font size="6"><strong>Act I: Inherited Decisions</strong></font></p>

<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/inherited-decisions.jpg" alt="A group of clinicians seen from behind or in profile against a teal background. Most wear white coats; one person, with darker skin, wears teal scrubs." width="640" />
  <p><em>The people in the room inherit the equation. None of them wrote it.</em></p>
</div>

---

## Start here
This notebook answers three questions:
1. How often does replacing the 2009 equation with the 2021 refit move adults into CKD?
2. Does that movement differ for Black and non-Black adults?
3. How does the answer change at other decision thresholds?

**Workflow:** predict -> run -> inspect -> interpret.

## Colab setup
Skip this cell if you are running locally.

In [ ]:
import sys, subprocess

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-deps", "--no-cache-dir",
        "data-supply-chain-workshop==0.1.2",
    ])
else:
    import data_supply_chain

## Cold start
Same cohort, same split, whichever notebook you opened first. Earlier
notebooks do not need to be open.

In [ ]:
from data_supply_chain.bootstrap import cold_start

S = cold_start()
print(f"Cohort loaded: {len(S.df):,} adults · source={S.source} · real={S.is_real}")

## **Scale from one patient to the cohort**
Now we test this across the full CDC NHANES cohort.

**Look at the 60 threshold first.**
How often do patients move into CKD when race is removed from the equation?

In [ ]:
# Demo, take this opportunity to review the output

from data_supply_chain.egfr import add_egfr_columns, reclass_table

df = add_egfr_columns(S.df)
tab = reclass_table(df)

print("Reclassification rates at key thresholds")
print(tab.round(2).to_string(index=False))

print("\nAt threshold 60 (G3a boundary):")
t60 = tab.loc[tab["threshold"] == 60].iloc[0]
black_pct_60 = float(t60["Black_pct"])
nonblack_pct_60 = float(t60["NonBlack_pct"])
print(f"  Black adults into CKD:     {black_pct_60:.2f}%")
print(f"  Non-Black adults into CKD: {nonblack_pct_60:.2f}%")
print(f"  Gap (Black - Non-Black):   {black_pct_60 - nonblack_pct_60:+.2f}%")

### **Attendee challenge: change the threshold**
Same cohort. Same two equations. Only the decision boundary changes.
Edit `challenge_threshold` in the next cell (try 45, 60, or 20) and re-run.

In [ ]:
# Your turn, change the threshold

# Edit this line; try 45, 60, or 20
challenge_threshold = 20

t_custom = tab.loc[tab["threshold"] == challenge_threshold].iloc[0]
black_pct_custom = float(t_custom["Black_pct"])
nonblack_pct_custom = float(t_custom["NonBlack_pct"])
print(f"Challenge boundary = {challenge_threshold} ({t_custom['label']})")
print(f"  Into CKD (Black):     {black_pct_custom:.2f}%")
print(f"  Into CKD (Non-Black): {nonblack_pct_custom:.2f}%")

Attendee Insight:

Record your thoughts here

---

## Maintainer cell (safe to skip in workshop)
This writes the findings the website reads, so the numbers on the page stay
synced to this notebook's output.

In [ ]:
from data_supply_chain.findings import Findings
from data_supply_chain.paths import findings_path

f = Findings.open()
for _, row in tab.iterrows():
    thr = int(row["threshold"])
    f.record(f"act1.reclass.t{thr}.black_pct", round(float(row["Black_pct"]), 2))
    f.record(f"act1.reclass.t{thr}.nonblack_pct", round(float(row["NonBlack_pct"]), 2))
    f.record(f"act1.reclass.t{thr}.label", row["label"])

if findings_path() is None:
    print("Outside a repo checkout - findings.json not written (expected in Colab).")
else:
    f.write(state=S)
    print("wrote findings.json")